# 📎 Detailed Cheat Sheet: Joining Tables & Funnel Analysis
### Reference notebook — pandas (Python) with dplyr (R) equivalents

Everything you need to join multiple tables and build a conversion funnel,
in one place. Run the setup cell first, then jump to whichever section you need.


## Setup

In [1]:
import pandas as pd

visits = pd.read_csv('data/visits.csv', parse_dates=['visit_time'])
cart = pd.read_csv('data/cart.csv', parse_dates=['cart_time'])
checkout = pd.read_csv('data/checkout.csv', parse_dates=['checkout_time'])
purchase = pd.read_csv('data/purchase.csv', parse_dates=['purchase_time'])
print("loaded:", len(visits), len(cart), len(checkout), len(purchase))

loaded: 2000 620 410 290


## 1. The join types at a glance

| Goal | pandas | dplyr (R) | Keeps |
|---|---|---|---|
| Only rows that match in both | `a.merge(b, how='inner')` | `a %>% inner_join(b)` | intersection |
| All of `a`, matched `b` | `a.merge(b, how='left')` | `a %>% left_join(b)` | all of left |
| All of `b`, matched `a` | `a.merge(b, how='right')` | `a %>% right_join(b)` | all of right |
| Everything, matched or not | `a.merge(b, how='outer')` | `a %>% full_join(b)` | union |
| Stack rows (same columns) | `pd.concat([a, b])` | `a %>% bind_rows(b)` | union of rows |

**Rule of thumb:** if you're asking *"who dropped off?"* or *"what's missing?"* — you want a
**left join** and then a filter for nulls. If you're asking *"what do these two datasets have
in common?"* — you want an **inner join**. If you're merging two *complete but separate*
customer lists and don't want to lose anyone — **full join**.

## 2. Inner join — only exact matches

In [2]:
# pandas
inner = cart.merge(checkout, how='inner', on='user_id')
print(len(inner), "users who both added to cart AND started checkout")

# dplyr equivalent:
#   inner <- cart %>% inner_join(checkout)

410 users who both added to cart AND started checkout


## 3. Left join — keep everything on the left, find what's missing

This is the workhorse of funnel analysis: it tells you who **didn't** make it
to the next step.

In [3]:
left = cart.merge(checkout, how='left', on='user_id')
dropped = left[left.checkout_time.isnull()]
pct = len(dropped) / len(left) * 100
print(f"{pct:.1f}% of cart-adders never reached checkout")

# dplyr equivalent:
#   left <- cart %>% left_join(checkout)
#   dropped <- left %>% filter(is.na(checkout_time))

33.9% of cart-adders never reached checkout


## 4. Right join — the mirror image of left join

In [4]:
right = cart.merge(checkout, how='right', on='user_id')
print(len(right), "== len(checkout) since every checkout user is kept:", len(checkout))

# dplyr equivalent:
#   right <- cart %>% right_join(checkout)
# NOTE: right_join(a, b) == left_join(b, a) with column order swapped.
# Most people just flip the argument order and use left_join instead.

410 == len(checkout) since every checkout user is kept: 410


## 5. Full (outer) join — lose nobody

In [5]:
# Toy example: two partial customer lists that need to be reconciled
company_a = pd.DataFrame({'name': ['Sally', 'Peter', 'Leslie'],
                           'email': ['sally@x.com', 'peter@x.com', 'leslie@x.com']})
company_b = pd.DataFrame({'name': ['Peter', 'Leslie', 'Aaron'],
                           'phone': ['212-555-0101', '626-555-0102', '303-555-0103']})

full = company_a.merge(company_b, how='outer', on='name')
full

# dplyr equivalent:
#   full <- company_a %>% full_join(company_b)

,name,email,phone
0,Aaron,NaN,303-555-0103
1,Leslie,leslie@x.com,626-555-0102
2,Peter,peter@x.com,212-555-0101
3,Sally,sally@x.com,NaN


## 6. Chaining multiple joins together

Use method chaining (`\` line continuation) to join more than two tables in one
readable expression. Left joins are almost always the right default for a funnel,
since you want to keep every original visitor even as you add later-stage columns.

In [6]:
all_data = visits.merge(cart, how='left', on='user_id') \
                  .merge(checkout, how='left', on='user_id') \
                  .merge(purchase, how='left', on='user_id')
all_data.shape

# dplyr equivalent:
#   all_data <- visits %>%
#     left_join(cart) %>%
#     left_join(checkout) %>%
#     left_join(purchase)

(2000, 5)

## 7. When column names don't match

If your join key is spelled differently in each table (`id` vs. `customer_id`),
tell `merge`/`inner_join` explicitly which columns correspond.

In [7]:
products = pd.DataFrame({'id': [1, 2, 3], 'description': ['tee', 'hoodie', 'cap']})
orders = pd.DataFrame({'order_id': [101, 102], 'product_id': [1, 3], 'qty': [2, 1]})

merged = orders.merge(products, how='left', left_on='product_id', right_on='id')
merged

# dplyr equivalent:
#   merged <- orders %>%
#     left_join(products, by = c('product_id' = 'id'))

,order_id,product_id,qty,id,description
0,101,1,2,1,tee
1,102,3,1,3,cap


## 8. Duplicate column names → suffixes

When both tables have a same-named column that *isn't* the join key, pandas
(and dplyr) auto-suffix them. Name your own suffixes for readability.

In [8]:
left_named = orders.merge(
    products, how='left', left_on='product_id', right_on='id',
    suffixes=('_order', '_product')
)
left_named

# dplyr equivalent:
#   left_named <- orders %>%
#     left_join(products, by = c('product_id' = 'id'),
#               suffix = c('_order', '_product'))

,order_id,product_id,qty,id,description
0,101,1,2,1,tee
1,102,3,1,3,cap


## 9. Concatenating (stacking) tables with the same columns

Different from a join — this is for when a dataset is *split across files*
with identical columns (e.g. monthly exports) and needs to be reassembled.

In [9]:
jan = pd.DataFrame({'month': ['Jan'], 'revenue': [300]})
feb = pd.DataFrame({'month': ['Feb'], 'revenue': [290]})
combined = pd.concat([jan, feb], ignore_index=True)
combined

# dplyr equivalent:
#   combined <- jan %>% bind_rows(feb)

,month,revenue
0,Jan,300
1,Feb,290


## 10. Funnel-analysis-specific recipes

These are the patterns you'll reuse on almost every funnel project.

**Recipe A — % drop-off between two adjacent stages**

In [10]:
def dropoff_percent(upstream_df, downstream_df, key='user_id'):
    merged = upstream_df.merge(downstream_df, how='left', on=key)
    # the downstream table's non-key column is whatever wasn't in upstream_df
    downstream_col = [c for c in downstream_df.columns if c != key][0]
    dropped = merged[merged[downstream_col].isnull()]
    return len(dropped) / len(merged) * 100

print(f"{dropoff_percent(visits, cart):.1f}% visit -> cart drop-off")
print(f"{dropoff_percent(cart, checkout):.1f}% cart -> checkout drop-off")
print(f"{dropoff_percent(checkout, purchase):.1f}% checkout -> purchase drop-off")

69.0% visit -> cart drop-off
33.9% cart -> checkout drop-off
29.3% checkout -> purchase drop-off


**Recipe B — time elapsed between two funnel events**

In [11]:
timed = visits.merge(purchase, how='inner', on='user_id')
timed['time_to_purchase'] = timed['purchase_time'] - timed['visit_time']
print("Average:", timed['time_to_purchase'].mean())
print("Median: ", timed['time_to_purchase'].median())

Average: 0 days 04:21:20.896551
Median:  0 days 04:27:00


**Recipe C — funnel counts table, ready for plotting**

In [12]:
funnel = pd.DataFrame({
    'stage': ['Visit', 'Cart', 'Checkout', 'Purchase'],
    'users': [len(visits), len(cart), len(checkout), len(purchase)],
})
funnel['conversion_from_start'] = (funnel['users'] / funnel['users'].iloc[0] * 100).round(1)
funnel

,stage,users,conversion_from_start
0,Visit,2000,100.0
1,Cart,620,31.0
2,Checkout,410,20.5
3,Purchase,290,14.5


## 11. Common pitfalls

- **Forgetting `how='left'`** — pandas' `.merge()` defaults to `how='inner'`, which
  silently *drops* everyone who didn't reach the next stage. If you're measuring
  drop-off, an inner join will hide the very thing you're trying to measure.
- **Joining on the wrong key after a rename** — always re-`.head()` a table right
  after renaming a column, before joining on it.
- **Duplicate join keys** — if a key isn't unique in either table, a join can
  silently multiply rows (a "fan-out"). Check `df['user_id'].is_unique` before
  joining if you're not sure.
- **Comparing counts across differently-filtered tables** — always compute
  percentages relative to the *same* denominator you intend (e.g. all visitors,
  not all rows in a joined table with duplicates).
- **NaT/NaN in time math** — subtracting timestamps where one side is missing
  produces `NaT`; filter or `.dropna()` before averaging.